# CircuitVision: baseline experiment on real CGHD data (baseline_v1)

Runtime → Change runtime type → **T4 GPU**. Run top to bottom.

Everything lands in `MyDrive/CircuitVision/experiments/baseline_v1`, so a disconnect loses nothing.

**Two stop points need a human:**
- Step 2: read the rotation sheets and confirm the convention.
- Step 3: look at the label overlays.

Do not continue past a stop point if what you see is wrong.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
EXP = '/content/drive/MyDrive/CircuitVision/experiments/baseline_v1'
DATA = '/content/drive/MyDrive/CircuitVision/data'
import os; os.makedirs(EXP, exist_ok=True); os.makedirs(DATA, exist_ok=True)

In [ ]:
%cd /content
!rm -rf CircuitVision && git clone -q https://github.com/sudo-pranshu/CircuitVision.git
%cd /content/CircuitVision
!pip install -q -e ".[detect]" && apt-get -qq install -y ngspice > /dev/null
!git rev-parse HEAD | tee {EXP}/commit.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Dataset: CGHD, Zenodo record 14042961 (v14, 4.4 GB, CC BY 3.0)
The zip is cached on Drive and checked against Zenodo's published MD5.

In [ ]:
ZIP = f'{DATA}/cghd-zenodo-14.zip'
if not os.path.exists(ZIP):
    !wget -q --show-progress -O {ZIP} 'https://zenodo.org/records/14042961/files/cghd-zenodo-14.zip?download=1'
!md5sum {ZIP} | tee {EXP}/dataset_md5.txt
!echo "expected: ab1cde6feb5edaafbde711cd2059a2f4"
!rm -rf /content/cghd && unzip -q -o {ZIP} -d /content/cghd
import glob
CGHD = os.path.dirname(sorted(glob.glob('/content/cghd/**/drafter_1', recursive=True))[0])
print('CGHD root:', CGHD)

## Step 1: dataset audit
This reads every XML and image and assumes nothing from the docs.

In [ ]:
!python -m circuitvision.audit --cghd {CGHD} --out {EXP}/audit
!head -c 3000 {EXP}/audit/raw_xml_samples.txt

## Step 2: rotation convention (STOP POINT)

Each sheet has one row per `<rotation>` value, with crops shown including wire context. For each row, write down:

- Which way the diode's triangle points (anode → cathode), or where the source's + is.
- Whether going from 0 to 90 turns the symbol clockwise or counter-clockwise.

Then set `PLUS_AT_0` and `CGHD_CLOCKWISE` in `src/circuitvision/orientation.py`. If they're already right, leave them. Send the sheets back to Claude either way.

In [ ]:
!python -m circuitvision.visualize rotation --cghd {CGHD} --out {EXP}/rotation_check
from IPython.display import Image, display
for p in sorted(glob.glob(f'{EXP}/rotation_check/*.png')):
    print(p); display(Image(p))

## Step 3: conversion and label check (STOP POINT)
The split is by drafter: validation = drafter_11, test = drafter_12, everything else is training.

In [ ]:
!python -m circuitvision.cghd_to_yolo --cghd {CGHD} --out /content/yolo --val-drafters 11 --test-drafters 12
!cp /content/yolo/conversion_stats.json {EXP}/
!python -m circuitvision.visualize labels --yolo /content/yolo --out {EXP}/label_check -n 24
for p in sorted(glob.glob(f'{EXP}/label_check/*.jpg'))[:8]:
    print(p); display(Image(p, width=900))

Look through all 24 overlays in `label_check/`. Boxes must sit on their symbols. A consistent shift or a 90° rotation means an EXIF or size problem, so stop and report it.

## Step 4: train the baseline YOLOv8s
The config is fixed in `train.py` (`BASELINE`). If Colab disconnects, re-run the resume cell.

In [ ]:
!python -m circuitvision.train --data /content/yolo/dataset.yaml --exp {EXP}

In [ ]:
# Only after a disconnect:
# !python -m circuitvision.train --data /content/yolo/dataset.yaml --exp {EXP} --resume

## Step 5: evaluate on test drafter 12

In [ ]:
BEST = f'{EXP}/runs/train/weights/best.pt'
!python -m circuitvision.detector_eval --weights {BEST} --data /content/yolo/dataset.yaml --exp {EXP}

## Step 6: downstream stages, ground-truth boxes vs detector boxes
Run this only after the rotation convention is confirmed in Step 2.

In [ ]:
!python -m circuitvision.orientation --cghd {CGHD} --out {EXP}/models --test-drafters 12 --exp {EXP}
!python -m circuitvision.evaluate --cghd {CGHD} --drafters 12 --exp {EXP} --orient {EXP}/models --ocr --weights {BEST}

## Evidence bundle for the report and for Claude
This collects small files only (JSON, MD, CSV, TXT and images) and leaves out checkpoints.

In [ ]:
!cd {EXP} && zip -q -r /content/baseline_v1_evidence.zip . -x 'runs/*/weights/*' -x '*.pt' -x 'models/*'
!ls -lh /content/baseline_v1_evidence.zip
from google.colab import files; files.download('/content/baseline_v1_evidence.zip')